# 01 — Dados e EDA: cancelamento de reservas de hotel

**Pergunta:** no momento em que a reserva é feita, dá para saber quais vão ser canceladas?

**Dados:** 119.390 reservas de dois hotéis em Portugal (City Hotel e Resort Hotel), chegadas de jul/2015 a ago/2017.
Fonte: Antonio, Almeida & Nunes (2019), *Hotel booking demand datasets*, Data in Brief 22. Baixado do Kaggle (`jessemostipak/hotel-booking-demand`).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_rows", 50)

df = pd.read_csv("../data/raw/hotel_bookings.csv")
df.shape

(119390, 32)

## Etapa 1 — Conhecer e limpar os dados

### 1.1 Visão geral das colunas

Para cada coluna: tipo, quantos nulos, quantos valores diferentes e um exemplo de valor.

In [2]:
resumo = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(1),
    "valores distintos": df.nunique(),
    "exemplo": df.apply(lambda col: col.dropna().iloc[0]),
})
resumo

,tipo,nulos,% nulos,valores distintos,exemplo
hotel,str,0,0.0,2,Resort Hotel
is_canceled,int64,0,0.0,2,0
lead_time,int64,0,0.0,479,342
arrival_date_year,int64,0,0.0,3,2015
arrival_date_month,str,0,0.0,12,July
arrival_date_week_number,int64,0,0.0,53,27
arrival_date_day_of_month,int64,0,0.0,31,1
stays_in_weekend_nights,int64,0,0.0,17,0
stays_in_week_nights,int64,0,0.0,35,0
adults,int64,0,0.0,14,2


### 1.2 O que cada coluna significa

Resumo do dicionário de dados do artigo original.

| Coluna | Significado |
|---|---|
| `hotel` | City Hotel (Lisboa) ou Resort Hotel (Algarve) |
| `is_canceled` | **Alvo.** 1 = reserva cancelada, 0 = não cancelada |
| `lead_time` | Dias entre a data em que a reserva foi feita e a data de chegada |
| `arrival_date_year`, `_month`, `_week_number`, `_day_of_month` | Data de chegada |
| `stays_in_weekend_nights` | Noites de fim de semana (sáb/dom) reservadas |
| `stays_in_week_nights` | Noites de semana (seg–sex) reservadas |
| `adults`, `children`, `babies` | Número de hóspedes por faixa |
| `meal` | Refeição: BB = café da manhã, HB = meia pensão, FB = pensão completa, SC/Undefined = sem refeição |
| `country` | País de origem do hóspede (código de 3 letras) |
| `market_segment` | Segmento de mercado (TA = agência de viagem, TO = operadora) |
| `distribution_channel` | Canal pelo qual a reserva chegou |
| `is_repeated_guest` | 1 = hóspede que já se hospedou antes |
| `previous_cancellations` | Reservas anteriores do cliente que foram canceladas |
| `previous_bookings_not_canceled` | Reservas anteriores do cliente que não foram canceladas |
| `reserved_room_type` | Tipo de quarto reservado (código anonimizado) |
| `assigned_room_type` | Tipo de quarto atribuído ao hóspede — pode diferir do reservado |
| `booking_changes` | Alterações feitas na reserva até o check-in ou o cancelamento |
| `deposit_type` | No Deposit, Non Refund (depósito do valor total, sem reembolso) ou Refundable (depósito parcial) |
| `agent` | ID da agência de viagem que fez a reserva |
| `company` | ID da empresa que fez a reserva ou que paga por ela |
| `days_in_waiting_list` | Dias na lista de espera até a reserva ser confirmada |
| `customer_type` | Transient (avulsa), Transient-Party (avulsa ligada a outra), Contract, Group |
| `adr` | Diária média: valor total da hospedagem ÷ número de noites |
| `required_car_parking_spaces` | Vagas de estacionamento pedidas |
| `total_of_special_requests` | Pedidos especiais (ex.: cama de casal, andar alto) |
| `reservation_status` | Último status: Canceled, Check-Out ou No-Show |
| `reservation_status_date` | Data do último status |

### 1.3 O que a visão geral mostra

**Nulos.** Só 4 colunas têm nulos, e nem todo nulo é "dado faltando":
- `company` (94% nulo) e `agent` (14% nulo): no arquivo original o valor é `NULL`, que significa *não se aplica* — a reserva não foi feita por empresa ou por agência. Não é falha de preenchimento.
- `country` (488 linhas, 0,4%): país não informado.
- `children` (4 linhas): dado faltando de fato.

**Tipos para ajustar na limpeza:**
- `agent` e `company` estão como número, mas são IDs — devem ser tratados como categoria.
- `children` virou decimal (`float`) só por causa dos 4 nulos.
- A data de chegada está espalhada em 4 colunas, com o mês em texto — vale montar uma coluna de data só.
- `reservation_status_date` está como texto.
- Em `meal`, `SC` e `Undefined` significam a mesma coisa (sem refeição).